In [1]:
# =============================
# CELL 1: IMPORTS + BASIC SETUP
# =============================

import os
import re
import json
import math
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

from openai import OpenAI
import google.generativeai as genai
import anthropic

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Imports loaded successfully")
print("Working directory:", os.getcwd())

C:\Users\panit\AppData\Local\Temp\ipykernel_11848\13136864.py:25: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Imports loaded successfully
Working directory: c:\Project_On_LLM\notebooks


In [2]:
# =====================================
# CELL 2: PROJECT PATHS + DATASET SETUP
# =====================================

# project root
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CORRUPT_DIR = PROJECT_ROOT / "data" / "corrupted"
BASELINE_DIR = PROJECT_ROOT / "data" / "cleaned_baseline"
LLM_DIR = PROJECT_ROOT / "data" / "cleaned_llm"

# create folders if missing
CORRUPT_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
LLM_DIR.mkdir(parents=True, exist_ok=True)

# datasets
FILES = {
    "blood": "blood.csv",
    "boston_airbnb": "boston_airbnb.csv",
    "credit_risk": "credit_risk.csv",
    "medical_appointments": "medical_appointments.csv",
    "telco_customer_churn": "telco_customer_churn.csv",
}

# targets
TARGETS = {
    "blood": "class",
    "boston_airbnb": "price",
    "credit_risk": "person_income",
    "medical_appointments": "age",
    "telco_customer_churn": "churn",
}

print("RAW_DIR:", RAW_DIR)
print("Datasets found:", os.listdir(RAW_DIR))

RAW_DIR: C:\Project_On_LLM\data\raw
Datasets found: ['blood.csv', 'boston_airbnb.csv', 'credit_risk.csv', 'medical_appointments.csv', 'telco_customer_churn.csv']


In [3]:
# =====================================
# CELL 3: COLUMN NORMALIZATION HELPERS
# =====================================

def normalize_colname(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_colname(c) for c in df.columns]
    return df


def coerce_numeric_series(s: pd.Series) -> pd.Series:

    if pd.api.types.is_numeric_dtype(s):
        return s

    x = s.astype(str).str.strip()

    x = x.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan
    })

    x = x.str.replace(r"[\$,]", "", regex=True)
    x = x.str.replace("%", "", regex=False)

    out = pd.to_numeric(x, errors="coerce")

    return out


def ensure_target_numeric_or_binary(df: pd.DataFrame, target: str) -> pd.DataFrame:

    df = df.copy()

    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found")

    y = df[target]

    y_num = coerce_numeric_series(y)

    if y_num.notna().mean() >= 0.8:
        df[target] = y_num
        df = df[df[target].notna()].copy()
        return df

    uniq = y.dropna().astype(str).unique()

    if len(uniq) == 2:
        mapping = {uniq[0]: 0, uniq[1]: 1}
        df[target] = y.astype(str).map(mapping)
        df = df[df[target].notna()].copy()
        return df

    raise ValueError(
        f"Target '{target}' is not numeric and not binary."
    )


print("Column normalization helpers loaded.")

Column normalization helpers loaded.


In [4]:
# =====================================
# CELL 4: LOAD RAW DATASETS SAFELY
# =====================================

raw_data = {}
dataset_summary = []

for name, fname in FILES.items():
    path = RAW_DIR / fname

    df = pd.read_csv(path)
    df = normalize_columns(df)

    target = normalize_colname(TARGETS[name])
    df = ensure_target_numeric_or_binary(df, target)

    raw_data[name] = df

    dataset_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "target": target,
        "missing_target": int(df[target].isna().sum())
    })

summary_df = pd.DataFrame(dataset_summary).sort_values("dataset").reset_index(drop=True)

print("Loaded datasets successfully.")
display(summary_df)

Loaded datasets successfully.


,dataset,rows,cols,target,missing_target
0,blood,748,5,class,0
1,boston_airbnb,3585,95,price,0
2,credit_risk,32581,12,person_income,0
3,medical_appointments,110527,14,age,0
4,telco_customer_churn,7043,21,churn,0


In [5]:
# =====================================
# CELL 5: COLUMN SELECTION HELPERS
# =====================================

def pick_columns(df: pd.DataFrame, protected_cols=None):

    protected_cols = set(protected_cols or [])

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    text_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    skip_pattern = re.compile(
        r"(id$|_id$|customerid|rowid|index|ssn|email|phone|zip|zipcode)",
        re.I
    )

    num_cols = [
        c for c in num_cols
        if (not skip_pattern.search(str(c))) and (c not in protected_cols)
    ]

    text_cols = [
        c for c in text_cols
        if (not skip_pattern.search(str(c))) and (c not in protected_cols)
    ]

    return num_cols, text_cols


print("Column selection helper ready.")

Column selection helper ready.


In [8]:
# =====================================
# CELL 6: DATA CORRUPTION HELPERS
# =====================================


MISSING_RATE = 0.10
DUPLICATE_RATE = 0.05
OUTLIER_RATE = 0.03
TEXT_NOISE_RATE = 0.05


def inject_missing(df, cols, rate):

    if not cols or rate <= 0:
        return df

    df2 = df.copy()

    n_rows = len(df2)
    n_cols = len(cols)

    k = max(1, int(n_rows * n_cols * rate))

    row_idx = rng.integers(0, n_rows, size=k)
    col_idx = rng.integers(0, n_cols, size=k)

    for r, ci in zip(row_idx, col_idx):
        col = cols[ci]
        df2.iat[r, df2.columns.get_loc(col)] = np.nan

    return df2


def inject_duplicates(df, rate):

    if rate <= 0 or len(df) < 2:
        return df

    k = max(1, int(len(df) * rate))

    dup_rows = df.sample(n=k, random_state=SEED)

    return pd.concat([df, dup_rows], ignore_index=True)


def inject_outliers(df, num_cols, rate):

    if not num_cols or rate <= 0:
        return df

    df2 = df.copy()

    n_rows = len(df2)
    k = max(1, int(n_rows * rate))

    for col in num_cols:

        rows = rng.choice(n_rows, size=min(k, n_rows), replace=False)

        s = df2[col].dropna()

        if s.empty:
            continue

        scale = s.std()

        if scale == 0 or pd.isna(scale):
            scale = (s.max() - s.min())

        if scale == 0 or pd.isna(scale):
            scale = 10

        for r in rows:

            if pd.isna(df2.at[r, col]):
                continue

            v = df2.at[r, col]

            df2.at[r, col] = v * 10 + 50 * scale

    return df2


def typo(val):

    if not isinstance(val, str) or len(val) == 0:
        return val

    choice = rng.integers(0, 3)

    if choice == 0:
        return val.upper()

    elif choice == 1 and len(val) >= 3:
        i = rng.integers(0, len(val))
        return val[:i] + val[i+1:]

    else:
        if len(val) < 2:
            return val

        i = rng.integers(0, len(val)-1)
        return val[:i] + val[i+1] + val[i] + val[i+2:]


def inject_text_noise(df, text_cols, rate):

    if not text_cols or rate <= 0:
        return df

    df2 = df.copy()

    n_rows = len(df2)
    n_cols = len(text_cols)

    k = max(1, int(n_rows * n_cols * rate))

    row_idx = rng.integers(0, n_rows, size=k)
    col_idx = rng.integers(0, n_cols, size=k)

    for r, ci in zip(row_idx, col_idx):

        col = text_cols[ci]

        val = df2.at[r, col]

        if pd.isna(val):
            continue

        df2.at[r, col] = typo(str(val))

    return df2


print("Corruption helpers ready.")

Corruption helpers ready.


In [9]:
# =====================================
# CELL 7: CREATE + SAVE CORRUPTED DATA
# =====================================

def corrupt_dataset(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    dfc = df.copy()

    # protect target from corruption
    num_cols, text_cols = pick_columns(dfc, protected_cols=[target_col])

    dfc = inject_missing(dfc, cols=(num_cols + text_cols), rate=MISSING_RATE)
    dfc = inject_text_noise(dfc, text_cols=text_cols, rate=TEXT_NOISE_RATE)
    dfc = inject_outliers(dfc, num_cols=num_cols, rate=OUTLIER_RATE)
    dfc = inject_duplicates(dfc, rate=DUPLICATE_RATE)

    return dfc


corrupt_info = []
corrupted_data = {}

for name, df in raw_data.items():
    target = normalize_colname(TARGETS[name])

    df_corrupt = corrupt_dataset(df, target_col=target)
    df_corrupt = normalize_columns(df_corrupt)
    df_corrupt = ensure_target_numeric_or_binary(df_corrupt, target)

    out_path = CORRUPT_DIR / f"{name}_corrupted.csv"
    df_corrupt.to_csv(out_path, index=False)

    corrupted_data[name] = df_corrupt

    corrupt_info.append({
        "dataset": name,
        "target": target,
        "raw_shape": df.shape,
        "corrupt_shape": df_corrupt.shape,
        "saved_to": str(out_path)
    })

corrupt_df = pd.DataFrame(corrupt_info).sort_values("dataset").reset_index(drop=True)

print("Corrupted datasets created and saved.")
display(corrupt_df)

C:\Users\panit\AppData\Local\Temp\ipykernel_11848\868708799.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
C:\Users\panit\AppData\Local\Temp\ipykernel_11848\868708799.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pyda

Corrupted datasets created and saved.


,dataset,target,raw_shape,corrupt_shape,saved_to
0,blood,class,"(748, 5)","(785, 5)",C:\Project_On_LLM\data\corrupted\blood_corrupt...
1,boston_airbnb,price,"(3585, 95)","(3764, 95)",C:\Project_On_LLM\data\corrupted\boston_airbnb...
2,credit_risk,person_income,"(32581, 12)","(34210, 12)",C:\Project_On_LLM\data\corrupted\credit_risk_c...
3,medical_appointments,age,"(110527, 14)","(116053, 14)",C:\Project_On_LLM\data\corrupted\medical_appoi...
4,telco_customer_churn,churn,"(7043, 21)","(7395, 21)",C:\Project_On_LLM\data\corrupted\telco_custome...


In [10]:
# =====================================
# CELL 8: BASELINE MODEL PIPELINE
# =====================================

def build_pipeline(num_cols, text_cols):

    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median"))
    ])

    text_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    pre = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("text", text_pipe, text_cols)
    ])

    model = Pipeline([
        ("prep", pre),
        ("reg", Ridge(alpha=1.0))
    ])

    return model


print("Baseline pipeline ready.")

Baseline pipeline ready.


In [11]:
# =====================================
# CELL 8: BASELINE CLEANING FUNCTIONS
# =====================================

def safe_str_series(s: pd.Series) -> pd.Series:
    out = s.astype(str)
    out = out.replace("nan", np.nan)
    return out


def duplicate_removal(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates().copy()


def text_standardization(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_cols = df.select_dtypes(include=["object", "category"]).columns

    for c in text_cols:
        df[c] = safe_str_series(df[c]).str.strip().str.lower()

    return df


def type_coercion(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame:
    df = df.copy()

    for c in df.columns:
        if df[c].dtype == "object":
            converted = pd.to_numeric(df[c], errors="coerce")
            if converted.notna().mean() >= threshold:
                df[c] = converted

    return df


def impute_simple(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].fillna(df[c].median())
        else:
            mode_vals = df[c].mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else "unknown"
            df[c] = df[c].fillna(fill_val)

    return df


def impute_distribution(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for c in df.columns:
        s = df[c]
        miss = s.isna()

        if not miss.any():
            continue

        if pd.api.types.is_numeric_dtype(s):
            pool = s.dropna().to_numpy()
            df.loc[miss, c] = rng.choice(pool, size=int(miss.sum()), replace=True) if pool.size else 0
        else:
            vc = s.dropna().astype(str).value_counts(normalize=True)
            if vc.empty:
                df.loc[miss, c] = "unknown"
            else:
                values = vc.index.to_list()
                probs = vc.values
                df.loc[miss, c] = rng.choice(values, size=int(miss.sum()), replace=True, p=probs)

    return df


def outliers_iqr_cap(df: pd.DataFrame, factor: float = 1.5) -> pd.DataFrame:
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns

    for c in num_cols:
        q1 = df[c].quantile(0.25)
        q3 = df[c].quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            continue

        lo = q1 - factor * iqr
        hi = q3 + factor * iqr
        df[c] = df[c].clip(lo, hi)

    return df


def outliers_zscore_cap(df: pd.DataFrame, z_thresh: float = 3.0) -> pd.DataFrame:
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns

    for c in num_cols:
        s = df[c]
        mu = s.mean()
        sigma = s.std()

        if pd.isna(sigma) or sigma == 0:
            continue

        z = (s - mu) / sigma
        df.loc[z > z_thresh, c] = mu + z_thresh * sigma
        df.loc[z < -z_thresh, c] = mu - z_thresh * sigma

    return df


def baseline_simple_impute(df: pd.DataFrame) -> pd.DataFrame:
    df = duplicate_removal(df)
    df = text_standardization(df)
    df = type_coercion(df)
    df = impute_simple(df)
    return df


def baseline_dist_impute(df: pd.DataFrame) -> pd.DataFrame:
    df = duplicate_removal(df)
    df = text_standardization(df)
    df = type_coercion(df)
    df = impute_distribution(df)
    return df


def baseline_iqr_cap(df: pd.DataFrame) -> pd.DataFrame:
    df = baseline_simple_impute(df)
    df = outliers_iqr_cap(df)
    return df


def baseline_zscore_cap(df: pd.DataFrame) -> pd.DataFrame:
    df = baseline_simple_impute(df)
    df = outliers_zscore_cap(df)
    return df


def baseline_logistic_detect_then_repair(df: pd.DataFrame) -> pd.DataFrame:
    df0 = duplicate_removal(df)
    df0 = text_standardization(df0)
    df0 = type_coercion(df0)

    num_cols = df0.select_dtypes(include=[np.number]).columns.tolist()

    if not num_cols:
        return impute_simple(df0)

    z_parts = []
    for c in num_cols:
        s = df0[c]
        mu = s.mean()
        sigma = s.std()

        if pd.isna(sigma) or sigma == 0:
            continue

        z_parts.append(((s - mu) / sigma).abs())

    if not z_parts:
        return impute_simple(df0)

    max_abs_z = pd.concat(z_parts, axis=1).max(axis=1)
    flagged = max_abs_z > 3

    df_fix = df0.copy()

    for c in num_cols:
        med = df_fix[c].median()
        df_fix.loc[flagged & df_fix[c].notna(), c] = med

    df_fix = impute_simple(df_fix)
    return df_fix


BASELINES = {
    "simple_impute": baseline_simple_impute,
    "dist_impute": baseline_dist_impute,
    "iqr_cap": baseline_iqr_cap,
    "zscore_cap": baseline_zscore_cap,
    "logistic_detect": baseline_logistic_detect_then_repair,
}

print("Baseline cleaning functions loaded.")

Baseline cleaning functions loaded.


In [23]:
# =====================================
# CELL 9: CROSS VALIDATION EVALUATION
# =====================================

def cv_mse(df, target):

    num_cols, text_cols = pick_columns(df, protected_cols=[target])

    X = df.drop(columns=[target])
    y = df[target]

    model = build_pipeline(num_cols, text_cols)

    cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_mean_squared_error"
    )

    mse = -scores

    return mse.mean(), mse.std()


print("Cross-validation evaluator ready.")

Cross-validation evaluator ready.


In [24]:
# =====================================
# CELL 10: RUN BASELINE EXPERIMENTS
# =====================================

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

pd.options.display.float_format = "{:,.4f}".format


def safe_str_series(s: pd.Series) -> pd.Series:
    out = s.astype(str)
    out = out.replace("nan", np.nan)
    return out


def text_standardization(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_cols = df.select_dtypes(include=["object", "category", "string"]).columns

    for c in text_cols:
        df[c] = safe_str_series(df[c]).str.strip().str.lower()

    return df


def outliers_zscore_cap(df: pd.DataFrame, z_thresh: float = 3.0):
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

        s = df[c]
        mu = s.mean()
        sigma = s.std()

        if pd.isna(sigma) or sigma == 0:
            continue

        upper = mu + z_thresh * sigma
        lower = mu - z_thresh * sigma

        df[c] = s.clip(lower, upper)

    return df


def baseline_zscore_cap(df):
    df = baseline_simple_impute(df)
    df = outliers_zscore_cap(df)
    return df


BASELINES = {
    "simple_impute": baseline_simple_impute,
    "dist_impute": baseline_dist_impute,
    "iqr_cap": baseline_iqr_cap,
    "zscore_cap": baseline_zscore_cap,
    "logistic_detect": baseline_logistic_detect_then_repair,
}


def build_pipeline_from_df(df, target):
    X = df.drop(columns=[target]).copy()

    all_missing_cols = [c for c in X.columns if X[c].isna().all()]
    if all_missing_cols:
        X = X.drop(columns=all_missing_cols)

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    text_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    pre = ColumnTransformer(
        [
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, text_cols)
        ],
        remainder="drop"
    )

    model = Ridge(alpha=1.0)

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", model)
    ])

    return X, pipe, all_missing_cols


def cv_mse(df, target):
    df = normalize_columns(df)
    df = ensure_target_numeric_or_binary(df, target)

    y = df[target]
    X, pipe, dropped_cols = build_pipeline_from_df(df, target)

    n_samples = len(X)
    if n_samples < 2:
        raise ValueError(f"Not enough samples after cleaning: n_samples={n_samples}")

    n_splits = min(5, n_samples)
    if n_splits < 2:
        raise ValueError(f"Need at least 2 samples for CV, got {n_samples}")

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    scores = cross_val_score(
        pipe,
        X,
        y,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )

    mse = -scores.mean()
    return float(mse), dropped_cols


baseline_rows = []

for name, df in corrupted_data.items():
    target = normalize_colname(TARGETS[name])

    for method_name, cleaner in BASELINES.items():
        try:
            df_clean = cleaner(df.copy())
            df_clean = normalize_columns(df_clean)
            df_clean = ensure_target_numeric_or_binary(df_clean, target)

            mean_mse, dropped_cols = cv_mse(df_clean, target)

            baseline_rows.append({
                "dataset": name,
                "model": method_name,
                "mse": mean_mse,
                "dropped_all_missing_cols": ", ".join(dropped_cols) if dropped_cols else ""
            })

            print(f"Done -> {name} | {method_name} | MSE={mean_mse:,.4f}")

        except Exception as e:
            print(f"Error -> {name} | {method_name} | {e}")

baseline_df = (
    pd.DataFrame(baseline_rows)
    .sort_values(["dataset", "mse"])
    .reset_index(drop=True)
)

print("\nBaseline experiments completed.")
display(baseline_df)

Done -> blood | simple_impute | MSE=0.1984
Done -> blood | dist_impute | MSE=0.1992
Done -> blood | iqr_cap | MSE=0.1831
Done -> blood | zscore_cap | MSE=0.1982
Done -> blood | logistic_detect | MSE=0.1669
Done -> boston_airbnb | simple_impute | MSE=16,571,088,069,968.6152
Done -> boston_airbnb | dist_impute | MSE=7,472,266,316.1498
Done -> boston_airbnb | iqr_cap | MSE=264,054,579.4869
Done -> boston_airbnb | zscore_cap | MSE=37,346,261,610.0946
Done -> boston_airbnb | logistic_detect | MSE=226,674,786.6353
Done -> credit_risk | simple_impute | MSE=3,835,155,866.0520
Done -> credit_risk | dist_impute | MSE=3,837,035,645.1776
Done -> credit_risk | iqr_cap | MSE=830,473,228.2409
Done -> credit_risk | zscore_cap | MSE=1,474,855,340.2167
Done -> credit_risk | logistic_detect | MSE=852,969,907.9571
Done -> medical_appointments | simple_impute | MSE=534.0849
Done -> medical_appointments | dist_impute | MSE=534.0849
Done -> medical_appointments | iqr_cap | MSE=534.0570
Done -> medical_appoin

,dataset,model,mse,dropped_all_missing_cols
0,blood,logistic_detect,0.1669,
1,blood,iqr_cap,0.1831,
2,blood,zscore_cap,0.1982,
3,blood,simple_impute,0.1984,
4,blood,dist_impute,0.1992,
5,boston_airbnb,logistic_detect,"226,674,786.6353","neighbourhood_group_cleansed, has_availability..."
6,boston_airbnb,iqr_cap,"264,054,579.4869","neighbourhood_group_cleansed, has_availability..."
7,boston_airbnb,dist_impute,"7,472,266,316.1498",
8,boston_airbnb,zscore_cap,"37,346,261,610.0946","neighbourhood_group_cleansed, has_availability..."
9,boston_airbnb,simple_impute,"16,571,088,069,968.6152","neighbourhood_group_cleansed, has_availability..."


In [25]:
# =====================================
# CELL 11: FORMATTED BASELINE RESULTS
# =====================================

baseline_pivot = baseline_df.pivot_table(
    index="dataset",
    columns="model",
    values="mse",
    aggfunc="mean"
)

baseline_pivot = baseline_pivot.sort_index(axis=1)

print("Baseline MSE comparison table:")
display(baseline_pivot)

baseline_best_df = (
    baseline_df.loc[baseline_df.groupby("dataset")["mse"].idxmin()]
    .sort_values("dataset")
    .reset_index(drop=True)
)

print("\nBest baseline method per dataset:")
display(baseline_best_df)

Baseline MSE comparison table:


model,dist_impute,iqr_cap,logistic_detect,simple_impute,zscore_cap
dataset,,,,,
blood,0.1992,0.1831,0.1669,0.1984,0.1982
boston_airbnb,"7,472,266,316.1498","264,054,579.4869","226,674,786.6353","16,571,088,069,968.6152","37,346,261,610.0946"
credit_risk,"3,837,035,645.1776","830,473,228.2409","852,969,907.9571","3,835,155,866.0520","1,474,855,340.2167"
medical_appointments,534.0849,534.0570,433.5750,534.0849,534.0270
telco_customer_churn,0.1552,0.1513,0.1456,0.1536,0.1536



Best baseline method per dataset:


,dataset,model,mse,dropped_all_missing_cols
0,blood,logistic_detect,0.1669,
1,boston_airbnb,logistic_detect,"226,674,786.6353","neighbourhood_group_cleansed, has_availability..."
2,credit_risk,iqr_cap,"830,473,228.2409",
3,medical_appointments,logistic_detect,433.5750,
4,telco_customer_churn,logistic_detect,0.1456,


In [47]:
# =====================================
# CELL 12: LOAD API KEYS + REGISTER LLMs
# =====================================

load_dotenv(PROJECT_ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

print("OPENAI loaded:", OPENAI_API_KEY is not None)
print("GEMINI loaded:", GEMINI_API_KEY is not None)
print("ANTHROPIC loaded:", ANTHROPIC_API_KEY is not None)

OPENAI_MODELS = [
    "gpt-4o-mini",
    "gpt-4.1-mini"
]

GEMINI_MODELS = [
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash"
]

CLAUDE_MODELS = [
    "claude-haiku-4-5",
    "claude-sonnet-4-6"
]

MODEL_REGISTRY = []

for m in OPENAI_MODELS:
    MODEL_REGISTRY.append(("openai", m))

for m in GEMINI_MODELS:
    MODEL_REGISTRY.append(("gemini", m))

for m in CLAUDE_MODELS:
    MODEL_REGISTRY.append(("claude", m))

print("\nRegistered models:")
for provider, model in MODEL_REGISTRY:
    print(f"{provider:7s} -> {model}")

OPENAI loaded: True
GEMINI loaded: True
ANTHROPIC loaded: True

Registered models:
openai  -> gpt-4o-mini
openai  -> gpt-4.1-mini
gemini  -> gemini-2.5-flash-lite
gemini  -> gemini-2.5-flash
claude  -> claude-haiku-4-5
claude  -> claude-sonnet-4-6


In [80]:
# =====================================
# CELL 13: MULTI-MODEL LLM CLEANING HELPERS (ROBUST)
# =====================================

import os
import json
import re
import numpy as np
import pandas as pd

from openai import OpenAI
import google.generativeai as genai
import anthropic


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df


def normalize_colname(col: str) -> str:
    return str(col).strip().lower().replace(" ", "_")


def clean_na_markers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.astype(object).where(pd.notna(df), np.nan)

    for c in df.columns:
        if (
            pd.api.types.is_object_dtype(df[c]) or
            pd.api.types.is_string_dtype(df[c]) or
            isinstance(df[c].dtype, pd.CategoricalDtype)
        ):
            df[c] = df[c].replace({
                "NA": np.nan,
                "<NA>": np.nan,
                "nan": np.nan,
                "NaN": np.nan,
                "None": np.nan,
                "null": np.nan,
                "NULL": np.nan,
                "": np.nan,
                " ": np.nan
            })

    return df


def coerce_numeric_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")

    s2 = s.astype(str).str.strip()
    s2 = s2.str.replace(",", "", regex=False)
    s2 = s2.str.replace("$", "", regex=False)
    s2 = s2.str.replace("%", "", regex=False)
    s2 = s2.str.replace(r"[^0-9\.\-]", "", regex=True)

    return pd.to_numeric(s2, errors="coerce")


def build_prompt_sample(df: pd.DataFrame, target_col: str, max_rows=10, max_cols=25):
    sample_df = df.head(max_rows).copy()
    sample_df = clean_na_markers(sample_df)

    cols = list(sample_df.columns)
    target_col = normalize_colname(target_col)

    numeric_cols = [c for c in cols if pd.api.types.is_numeric_dtype(sample_df[c])]
    text_cols = [c for c in cols if c not in numeric_cols]

    safe_text_cols = []
    for c in text_cols:
        s = sample_df[c].dropna().astype(str)
        avg_len = s.str.len().mean() if len(s) > 0 else 0

        # skip long free-text columns that often break JSON
        if avg_len <= 60:
            safe_text_cols.append(c)

    ordered_cols = []

    if target_col in cols:
        ordered_cols.append(target_col)

    for c in numeric_cols:
        if c not in ordered_cols:
            ordered_cols.append(c)

    for c in safe_text_cols:
        if c not in ordered_cols:
            ordered_cols.append(c)

    ordered_cols = ordered_cols[:max_cols]
    sample_df = sample_df[ordered_cols].copy()

    for c in sample_df.columns:
        if (
            pd.api.types.is_object_dtype(sample_df[c]) or
            pd.api.types.is_string_dtype(sample_df[c]) or
            isinstance(sample_df[c].dtype, pd.CategoricalDtype)
        ):
            sample_df[c] = sample_df[c].apply(
                lambda x: str(x)[:80] if pd.notna(x) else None
            )

    sample_df = sample_df.where(pd.notna(sample_df), None)
    return sample_df


def dataframe_to_prompt(sample_df: pd.DataFrame, target_col: str):
    data_json = sample_df.to_dict(orient="records")
    columns_json = list(sample_df.columns)

    prompt = f"""
You are a data cleaning assistant.

Clean the following tabular dataset sample by fixing:
- missing values
- obvious outliers
- inconsistent categorical values
- formatting issues

Rules:
- Preserve exactly the same columns and column order shown below
- Preserve the same number of rows
- Do not invent new columns
- Keep the target column present: {target_col}
- If a value is already valid, keep it unchanged
- For text/categorical columns, return plain text values only
- For numeric columns, return numeric values only when appropriate
- Return only valid JSON in this exact format:
{{"rows": [{{...}}, {{...}}]}}

Columns:
{json.dumps(columns_json, ensure_ascii=False)}

Dataset sample:
{json.dumps(data_json, ensure_ascii=False)}
"""
    return prompt.strip()


def extract_rows_object(text: str):
    if text is None:
        raise ValueError("Empty model response")

    s = text.strip()
    s = re.sub(r"^```json\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^```\s*", "", s)
    s = re.sub(r"\s*```$", "", s)

    obj = json.loads(s)

    if not isinstance(obj, dict):
        raise ValueError("Model response is not a JSON object.")

    if "rows" not in obj:
        raise ValueError("Model response missing 'rows' key.")

    rows = obj["rows"]

    if not isinstance(rows, list):
        raise ValueError("'rows' is not a list.")

    return rows


def sanitize_llm_output_types(cleaned_sample_df: pd.DataFrame, reference_df: pd.DataFrame) -> pd.DataFrame:
    cleaned_sample_df = cleaned_sample_df.copy()

    for c in cleaned_sample_df.columns:
        cleaned_sample_df[c] = cleaned_sample_df[c].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict, tuple, set)) else x
        )

        if c in reference_df.columns and pd.api.types.is_numeric_dtype(reference_df[c]):
            cleaned_sample_df[c] = pd.to_numeric(cleaned_sample_df[c], errors="coerce")
        else:
            cleaned_sample_df[c] = cleaned_sample_df[c].astype(object)

    return cleaned_sample_df


def align_series_to_reference_dtype(series: pd.Series, ref_series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(ref_series):
        return pd.to_numeric(series, errors="coerce")

    if pd.api.types.is_datetime64_any_dtype(ref_series):
        return pd.to_datetime(series, errors="coerce")

    out = series.copy()
    out = out.apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict, tuple, set)) else x
    )
    out = out.apply(lambda x: str(x) if pd.notna(x) else np.nan)

    return out.astype(object)


def finalize_cleaned_dataframe(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    df = df.copy()
    df = normalize_columns(df)
    target_col = normalize_colname(target_col)

    for c in df.columns:
        if (
            pd.api.types.is_string_dtype(df[c]) or
            isinstance(df[c].dtype, pd.CategoricalDtype)
        ):
            df[c] = df[c].astype(object)

    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            coerced = coerce_numeric_series(df[c])
            if coerced.notna().mean() >= 0.8:
                df[c] = coerced

    for c in df.columns:
        if df[c].isna().sum() == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[c]):
            med = df[c].median()
            if pd.isna(med):
                med = 0
            df[c] = df[c].fillna(med)
        else:
            mode = df[c].mode(dropna=True)
            fill_value = mode.iloc[0] if len(mode) > 0 else "missing"
            df[c] = df[c].fillna(fill_value)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' missing.")

    s = df[target_col]

    if not pd.api.types.is_numeric_dtype(s):
        numeric_try = coerce_numeric_series(s)
        if numeric_try.notna().mean() >= 0.7:
            med = numeric_try.median()
            if pd.isna(med):
                med = 0
            df[target_col] = numeric_try.fillna(med)
        else:
            lower_vals = s.astype(str).str.strip().str.lower()
            binary_map = {
                "yes": 1, "no": 0,
                "true": 1, "false": 0,
                "1": 1, "0": 0,
                "y": 1, "n": 0,
                "churn": 1, "not_churn": 0,
                "t": 1, "f": 0
            }
            mapped = lower_vals.map(binary_map)

            if mapped.notna().mean() >= 0.7:
                mode = mapped.mode(dropna=True)
                fill_value = int(mode.iloc[0]) if len(mode) > 0 else 0
                df[target_col] = mapped.fillna(fill_value).astype(int)

    if df.isna().sum().sum() > 0:
        raise ValueError("Final dataframe still contains NaN values.")

    return df


def _clean_json_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def _validate_json_text(text: str) -> str:
    cleaned = _clean_json_text(text)
    json.loads(cleaned)
    return cleaned


def get_llm_json_response(provider: str, model_name: str, prompt: str) -> str:
    if provider == "openai":
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You must return valid JSON only. "
                        "Return exactly one JSON object with key 'rows'. "
                        "Example: {\"rows\": [{...}, {...}]}"
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "cleaned_rows_payload",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "rows": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "additionalProperties": True
                                }
                            }
                        },
                        "required": ["rows"],
                        "additionalProperties": False
                    }
                }
            }
        )
        return response.choices[0].message.content

    if provider == "gemini":
        genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(
            [
                "Return only valid JSON with exact format "
                "{\"rows\": [{...}, {...}]}. No markdown. No explanation.",
                prompt
            ]
        )
        return response.text

    if provider == "claude":
        client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

        base_system = (
            "You are a JSON generator. "
            "Output valid JSON only. "
            "No markdown. No comments. No explanations."
        )

        base_user = f"""
Return exactly one JSON object in this format:
{{"rows": [{{...}}, {{...}}]}}

Important rules:
- Escape all double quotes inside string values properly
- Escape newline characters inside strings
- Do not leave any string unterminated
- Preserve the same columns and row count
- If a text field is long, still return it as a valid escaped JSON string

{prompt}
""".strip()

        response = client.messages.create(
            model=model_name,
            max_tokens=4000,
            temperature=0,
            system=base_system,
            messages=[{"role": "user", "content": base_user}]
        )

        text = response.content[0].text

        try:
            return _validate_json_text(text)
        except Exception:
            retry_user = f"""
Your previous response was not valid JSON.

Fix it and return exactly one valid JSON object with this schema:
{{"rows": [{{...}}, {{...}}]}}

Non-negotiable rules:
- Output JSON only
- No markdown fences
- No extra text
- Escape all internal quotes in strings
- Escape all newline characters in strings
- Every string must be properly closed
- Preserve the same columns and same number of rows
- Keep long text values as properly escaped JSON strings

Use this original cleaning task again:

{prompt}
""".strip()

            retry_response = client.messages.create(
                model=model_name,
                max_tokens=4000,
                temperature=0,
                system=base_system,
                messages=[{"role": "user", "content": retry_user}]
            )

            retry_text = retry_response.content[0].text
            return _validate_json_text(retry_text)

    raise ValueError(f"Unknown provider: {provider}")


def llm_clean_dataframe(provider, model_name, df: pd.DataFrame, target_col: str):
    original_df = df.copy()
    original_df = normalize_columns(original_df)
    prompt_df = clean_na_markers(original_df.copy())

    for c in original_df.columns:
        if (
            pd.api.types.is_string_dtype(original_df[c]) or
            isinstance(original_df[c].dtype, pd.CategoricalDtype)
        ):
            original_df[c] = original_df[c].astype(object)

    prompt_sample_df = build_prompt_sample(prompt_df, target_col=target_col, max_rows=10, max_cols=25)
    prompt = dataframe_to_prompt(prompt_sample_df, target_col=target_col)

    text = get_llm_json_response(provider, model_name, prompt)
    cleaned_rows = extract_rows_object(text)
    cleaned_sample_df = pd.DataFrame(cleaned_rows)

    if cleaned_sample_df.empty:
        raise ValueError("Model returned empty cleaned dataframe.")

    expected_cols = list(prompt_sample_df.columns)

    # fill any missing prompted columns from original prompt sample instead of failing
    for c in expected_cols:
        if c not in cleaned_sample_df.columns:
            cleaned_sample_df[c] = prompt_sample_df[c].values[:len(cleaned_sample_df)]

    cleaned_sample_df = cleaned_sample_df[expected_cols]
    cleaned_sample_df = clean_na_markers(cleaned_sample_df)
    cleaned_sample_df = sanitize_llm_output_types(cleaned_sample_df, prompt_sample_df[expected_cols])

    final_df = original_df.copy()
    n_rows_replace = min(len(cleaned_sample_df), len(final_df), 10)

    for c in expected_cols:
        replacement = cleaned_sample_df.loc[:n_rows_replace - 1, c].copy()
        reference = final_df.loc[:n_rows_replace - 1, c].copy()

        replacement = align_series_to_reference_dtype(replacement, reference)

        if (
            pd.api.types.is_string_dtype(final_df[c]) or
            pd.api.types.is_object_dtype(final_df[c]) or
            isinstance(final_df[c].dtype, pd.CategoricalDtype)
        ):
            final_df[c] = final_df[c].astype(object)

        final_df.loc[:n_rows_replace - 1, c] = replacement.values

    final_df = clean_na_markers(final_df)
    final_df = finalize_cleaned_dataframe(final_df, target_col)

    return final_df


print("Cell 13 loaded successfully.")

Cell 13 loaded successfully.


In [81]:
# =====================================
# CELL 14A: RUN ALL LLM MODELS
# =====================================

ensure_dir(str(LLM_DIR))

llm_results = []
model_outputs = {}

DATASET_PATHS = {ds: str(RAW_DIR / fname) for ds, fname in FILES.items()}
TARGETS_NORM = {k: normalize_colname(v) for k, v in TARGETS.items()}

for provider, model_name in MODEL_REGISTRY:
    model_key = f"{provider}:{model_name}"
    model_outputs[model_key] = {}

    print(f"\n{'='*70}")
    print(f"RUNNING MODEL: {model_key}")
    print(f"{'='*70}")

    for ds, input_path in DATASET_PATHS.items():
        target_col = TARGETS_NORM[ds]

        print(f"\n--- Dataset: {ds} ---")
        print(f"Input:  {input_path}")
        print(f"Target: {target_col}")

        safe_model_name = model_key.replace(":", "__").replace("/", "_")
        output_path = os.path.join(str(LLM_DIR), f"{ds}__{safe_model_name}.csv")

        try:
            df = pd.read_csv(input_path)
            df = normalize_columns(df)

            cleaned_df = llm_clean_dataframe(provider, model_name, df, target_col)
            cleaned_df.to_csv(output_path, index=False)

            llm_results.append({
                "dataset": ds,
                "model": model_key,
                "output_path": output_path,
                "status": "success",
                "error": None,
                "rows": len(cleaned_df),
                "cols": len(cleaned_df.columns)
            })

            model_outputs[model_key][ds] = output_path

            print("Status: success")
            print(f"Saved:  {output_path}")

        except Exception as e:
            llm_results.append({
                "dataset": ds,
                "model": model_key,
                "output_path": None,
                "status": "failed",
                "error": str(e),
                "rows": None,
                "cols": None
            })

            print("Status: failed")
            print(f"Error:  {e}")

llm_df = pd.DataFrame(llm_results)
display(llm_df)

print("\nCompleted model runs.")
print("Successful outputs summary:")
for model_name, outputs in model_outputs.items():
    print(f"{model_name}: {len(outputs)} datasets completed")


RUNNING MODEL: openai:gpt-4o-mini

--- Dataset: blood ---
Input:  C:\Project_On_LLM\data\raw\blood.csv
Target: class
Status: success
Saved:  C:\Project_On_LLM\data\cleaned_llm\blood__openai__gpt-4o-mini.csv

--- Dataset: boston_airbnb ---
Input:  C:\Project_On_LLM\data\raw\boston_airbnb.csv
Target: price
Status: success
Saved:  C:\Project_On_LLM\data\cleaned_llm\boston_airbnb__openai__gpt-4o-mini.csv

--- Dataset: credit_risk ---
Input:  C:\Project_On_LLM\data\raw\credit_risk.csv
Target: person_income
Status: success
Saved:  C:\Project_On_LLM\data\cleaned_llm\credit_risk__openai__gpt-4o-mini.csv

--- Dataset: medical_appointments ---
Input:  C:\Project_On_LLM\data\raw\medical_appointments.csv
Target: age
Status: success
Saved:  C:\Project_On_LLM\data\cleaned_llm\medical_appointments__openai__gpt-4o-mini.csv

--- Dataset: telco_customer_churn ---
Input:  C:\Project_On_LLM\data\raw\telco_customer_churn.csv
Target: churn
Status: success
Saved:  C:\Project_On_LLM\data\cleaned_llm\telco_cu

,dataset,model,output_path,status,error,rows,cols
0,blood,openai:gpt-4o-mini,C:\Project_On_LLM\data\cleaned_llm\blood__open...,success,None,748,5
1,boston_airbnb,openai:gpt-4o-mini,C:\Project_On_LLM\data\cleaned_llm\boston_airb...,success,None,3585,95
2,credit_risk,openai:gpt-4o-mini,C:\Project_On_LLM\data\cleaned_llm\credit_risk...,success,None,32581,12
3,medical_appointments,openai:gpt-4o-mini,C:\Project_On_LLM\data\cleaned_llm\medical_app...,success,None,110527,14
4,telco_customer_churn,openai:gpt-4o-mini,C:\Project_On_LLM\data\cleaned_llm\telco_custo...,success,None,7043,21
5,blood,openai:gpt-4.1-mini,C:\Project_On_LLM\data\cleaned_llm\blood__open...,success,None,748,5
6,boston_airbnb,openai:gpt-4.1-mini,C:\Project_On_LLM\data\cleaned_llm\boston_airb...,success,None,3585,95
7,credit_risk,openai:gpt-4.1-mini,C:\Project_On_LLM\data\cleaned_llm\credit_risk...,success,None,32581,12
8,medical_appointments,openai:gpt-4.1-mini,C:\Project_On_LLM\data\cleaned_llm\medical_app...,success,None,110527,14
9,telco_customer_churn,openai:gpt-4.1-mini,C:\Project_On_LLM\data\cleaned_llm\telco_custo...,success,None,7043,21



Completed model runs.
Successful outputs summary:
openai:gpt-4o-mini: 5 datasets completed
openai:gpt-4.1-mini: 5 datasets completed
gemini:gemini-2.5-flash-lite: 5 datasets completed
gemini:gemini-2.5-flash: 5 datasets completed
claude:claude-haiku-4-5: 5 datasets completed
claude:claude-sonnet-4-6: 5 datasets completed


In [85]:
# =====================================
# CELL 15: COMPUTE MSE FOR ALL MODELS (MEMORY-SAFE)
# =====================================

experiment_rows = []

def prepare_for_modeling(df: pd.DataFrame, target: str):

    df = df.copy()
    df = normalize_columns(df)
    target = normalize_colname(target)

    df = df.replace({pd.NA: np.nan})
    df = ensure_target_numeric_or_binary(df, target)

    # fill missing values
    for c in df.columns:
        if df[c].isna().sum() == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[c]):
            med = df[c].median()
            if pd.isna(med):
                med = 0
            df[c] = df[c].fillna(med)
        else:
            mode = df[c].mode(dropna=True)
            fill_val = mode.iloc[0] if len(mode) > 0 else "missing"
            df[c] = df[c].fillna(fill_val)

    return df


def reduce_feature_space(X: pd.DataFrame, max_categories=20):
    X = X.copy()

    drop_cols = []

    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            nunique = X[c].nunique(dropna=True)

            # drop extremely high-cardinality columns
            if nunique > max_categories:
                drop_cols.append(c)
            else:
                X[c] = X[c].astype(str)

    if drop_cols:
        X = X.drop(columns=drop_cols)

    X = pd.get_dummies(X, drop_first=True)
    X = X.replace({pd.NA: np.nan}).fillna(0)

    return X


def cv_mse_simple(df, target):

    from sklearn.model_selection import KFold
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error

    X = df.drop(columns=[target]).copy()
    y = pd.to_numeric(df[target], errors="coerce").fillna(0)

    X = reduce_feature_space(X, max_categories=20)

    model = RandomForestRegressor(
        n_estimators=20,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    )

    kf = KFold(n_splits=3, shuffle=True, random_state=42)

    mses = []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mse = mean_squared_error(y_test, preds)
        mses.append(mse)

    return np.mean(mses)


for model, outputs in model_outputs.items():
    for ds, path in outputs.items():
        try:
            target = normalize_colname(TARGETS[ds])

            df = pd.read_csv(path)
            df = prepare_for_modeling(df, target)

            mse = cv_mse_simple(df, target)

            experiment_rows.append({
                "dataset": ds,
                "model": model,
                "mse": mse
            })

        except Exception as e:
            experiment_rows.append({
                "dataset": ds,
                "model": model,
                "mse": np.nan,
                "error": str(e)
            })

experiment_df = pd.DataFrame(experiment_rows)
display(experiment_df)

print("\nMSE Comparison Table")
pivot_table = experiment_df.pivot(index="dataset", columns="model", values="mse")
display(pivot_table)

,dataset,model,mse
0,blood,openai:gpt-4o-mini,0.1718
1,boston_airbnb,openai:gpt-4o-mini,"16,832.8358"
2,credit_risk,openai:gpt-4o-mini,"984,667,824.7932"
3,medical_appointments,openai:gpt-4o-mini,368.4040
4,telco_customer_churn,openai:gpt-4o-mini,0.1397
5,blood,openai:gpt-4.1-mini,0.1711
6,boston_airbnb,openai:gpt-4.1-mini,"16,836.3436"
7,credit_risk,openai:gpt-4.1-mini,"984,667,824.7932"
8,medical_appointments,openai:gpt-4.1-mini,368.4075
9,telco_customer_churn,openai:gpt-4.1-mini,0.1398



MSE Comparison Table


model,claude:claude-haiku-4-5,claude:claude-sonnet-4-6,gemini:gemini-2.5-flash,gemini:gemini-2.5-flash-lite,openai:gpt-4.1-mini,openai:gpt-4o-mini
dataset,,,,,,
blood,0.1718,0.1718,0.1718,0.1718,0.1711,0.1718
boston_airbnb,"16,836.3436","16,836.3436","16,910.8971","17,132.4530","16,836.3436","16,832.8358"
credit_risk,"984,667,824.7932","984,667,824.7932","984,667,824.7932","984,667,824.7932","984,667,824.7932","984,667,824.7932"
medical_appointments,368.4075,368.4075,368.4075,368.4075,368.4075,368.4040
telco_customer_churn,0.1397,0.1397,0.1397,0.1397,0.1398,0.1397


In [94]:
# =====================================
# CELL 16: COMBINED FINAL TABLE
# BASELINE METHODS FIRST, THEN LLM MODELS
# =====================================

baseline_cols = list(baseline_pivot.columns)
llm_cols = list(pivot_table.columns)

final_combined_table = pd.concat([baseline_pivot, pivot_table], axis=1)
final_combined_table = final_combined_table[baseline_cols + llm_cols]

print("Final Combined Comparison Table:")
display(final_combined_table)

Final Combined Comparison Table:


model,dist_impute,iqr_cap,logistic_detect,simple_impute,zscore_cap,claude:claude-haiku-4-5,claude:claude-sonnet-4-6,gemini:gemini-2.5-flash,gemini:gemini-2.5-flash-lite,openai:gpt-4.1-mini,openai:gpt-4o-mini
dataset,,,,,,,,,,,
blood,0.1992,0.1831,0.1669,0.1984,0.1982,0.1718,0.1718,0.1718,0.1718,0.1711,0.1718
boston_airbnb,"7,472,266,316.1498","264,054,579.4869","226,674,786.6353","16,571,088,069,968.6152","37,346,261,610.0946","16,836.3436","16,836.3436","16,910.8971","17,132.4530","16,836.3436","16,832.8358"
credit_risk,"3,837,035,645.1776","830,473,228.2409","852,969,907.9571","3,835,155,866.0520","1,474,855,340.2167","984,667,824.7932","984,667,824.7932","984,667,824.7932","984,667,824.7932","984,667,824.7932","984,667,824.7932"
medical_appointments,534.0849,534.0570,433.5750,534.0849,534.0270,368.4075,368.4075,368.4075,368.4075,368.4075,368.4040
telco_customer_churn,0.1552,0.1513,0.1456,0.1536,0.1536,0.1397,0.1397,0.1397,0.1397,0.1398,0.1397


In [103]:
# BEST METHOD PER DATASET
winner_df = pd.DataFrame({
    "best_method": final_combined_table.idxmin(axis=1),
    "best_mse": final_combined_table.min(axis=1)
})

display(winner_df)

,best_method,best_mse
dataset,,
blood,logistic_detect,0.1669
boston_airbnb,openai:gpt-4o-mini,"16,832.8358"
credit_risk,iqr_cap,"830,473,228.2409"
medical_appointments,openai:gpt-4o-mini,368.4040
telco_customer_churn,claude:claude-haiku-4-5,0.1397
